# 02 Preprocessing and Feature Engineering

This notebook creates the preprocessed, modeling-ready dataset used in the thesis.

The temporal train/test split is **not** performed here. The output of this notebook is an unsplit dataset containing the final modeling predictors, the binary target, the subgroup variable, and the temporal split variable. Later notebooks use this file for feature diagnostics, temporal splitting, and model training.

Main steps:

1. Load the raw Kaggle/Crunchbase startup dataset.
2. Keep only `acquired` and `closed` firms and create the binary target.
3. Convert date and funding variables to usable data types.
4. Remove chronological inconsistencies.
5. Apply the 1990 cutoff to `first_funding_at` and imputed `founded_year`.
6. Handle missing values in `founded_year`, `country_code`, and `market`.
7. Create engineered funding-dynamics, stage-structure, and subgroup features.
8. Exclude identifier, leakage-risk, sparse, granular-location, and overlapping text variables.
9. Apply `log1p` transformation to retained funding amount variables.
10. Save the final unsplit modeling dataset and preprocessing summary tables.

## 1. Imports and paths

This section imports the required libraries and defines repository-style input and output paths.

In [48]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.2f}".format)

# Expected project structure:
# thesis-startup-success-prediction/
# ├── data/raw/investments_VC.csv
# ├── data/processed/
# ├── notebooks/02_preprocessing_feature_engineering.ipynb
# └── outputs/

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CSV_PATH = PROJECT_DIR / "data" / "raw" / "investments_VC.csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PREPROCESSING_TABLE_DIR = PROJECT_DIR / "outputs" / "tables" / "preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSING_TABLE_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESSED_DATA_PATH = PROCESSED_DIR / "startup_preprocessed_unsplit.csv"
PREPROCESSING_STEPS_PATH = PREPROCESSING_TABLE_DIR / "preprocessing_step_summary.csv"
ZERO_SHARE_PATH = PREPROCESSING_TABLE_DIR / "funding_zero_share_after_preprocessing.csv"
FEATURE_BLOCKS_PATH = PREPROCESSING_TABLE_DIR / "final_feature_blocks.csv"

if not RAW_CSV_PATH.exists():
    raise FileNotFoundError(f"Raw CSV file not found: {RAW_CSV_PATH}")

print("Setup completed.")

Setup completed.


## 2. Configuration

This section defines constants used throughout preprocessing, including status labels, date variables, and feature lists.

In [49]:
VALID_STATUSES = ["acquired", "closed"]
TARGET_MAP = {"acquired": 1, "closed": 0}

MIN_FIRST_FUNDING_YEAR = 1990
MIN_FOUNDING_YEAR = 1990

DATE_COLUMNS = ["founded_at", "first_funding_at", "last_funding_at"]

FUNDING_AMOUNT_COLUMNS = [
    "funding_total_usd",
    "seed", "venture", "angel", "grant",
    "debt_financing", "private_equity", "undisclosed",
    "equity_crowdfunding", "convertible_note", "product_crowdfunding",
    "round_a", "round_b", "round_c", "round_d", "round_e", "round_f", "round_g", "round_h",
    "post_ipo_equity", "post_ipo_debt", "secondary_market",
]

# Retained funding-scale predictors in the final feature set.
LOG1P_COLUMNS = [
    "funding_total_usd", "seed", "venture", "angel",
    "debt_financing", "private_equity", "undisclosed",
]

IDENTIFIER_COLUMNS = ["permalink", "name", "homepage_url"]
LEAKAGE_RISK_COLUMNS = ["post_ipo_equity", "post_ipo_debt", "secondary_market"]
SPARSE_EXCLUDED_COLUMNS = [
    "product_crowdfunding",
    "equity_crowdfunding",
    "round_h",
    "round_g",
    "round_f",
    "convertible_note",
    "grant",
]
RAW_DATE_NOT_FINAL_FEATURES = ["founded_at", "first_funding_at", "founded_month", "founded_quarter"]
GRANULAR_LOCATION_COLUMNS = ["state_code", "region", "city"]
TEXT_OVERLAP_COLUMNS = ["category_list"]

TARGET_COLUMN = "target"
SUBGROUP_COLUMN = "is_usa"
TEMPORAL_SPLIT_COLUMN = "last_funding_at"

## 3. Feature blocks used in the thesis

This section defines the five final feature groups used for the feature-group ablation analysis.

In [50]:
funding_scale_features = [
    "funding_total_usd", "seed", "venture", "angel",
    "debt_financing", "private_equity", "undisclosed",
]

funding_dynamics_features = [
    "founded_year", "funding_rounds",
    "funding_duration_days", "avg_days_between_rounds", "is_single_round",
]

stage_structure_features = [
    "has_round_a", "has_round_b", "has_round_c", "has_round_d", "has_round_e",
]

geography_features = ["country_code"]
industry_features = ["market"]

full_features = (
    funding_scale_features
    + funding_dynamics_features
    + stage_structure_features
    + geography_features
    + industry_features
)

feature_blocks = {
    "Funding scale": funding_scale_features,
    "Funding dynamics": funding_dynamics_features,
    "Stage structure": stage_structure_features,
    "Geography": geography_features,
    "Industry": industry_features,
}

feature_block_table = pd.DataFrame(
    [
        {"feature_group": group, "feature": feature}
        for group, features in feature_blocks.items()
        for feature in features
    ]
)

display(feature_block_table)

,feature_group,feature
0,Funding scale,funding_total_usd
1,Funding scale,seed
2,Funding scale,venture
3,Funding scale,angel
4,Funding scale,debt_financing
5,Funding scale,private_equity
6,Funding scale,undisclosed
7,Funding dynamics,founded_year
8,Funding dynamics,funding_rounds
9,Funding dynamics,funding_duration_days


## 4. Helper functions

This section defines small utility functions for column cleaning, numeric conversion, and preprocessing-step tracking.

In [51]:
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names so accidental spaces or case differences do not break later code."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def clean_money_like_series(series: pd.Series) -> pd.Series:
    """Convert strings such as '1,000' or '$1,000' to numeric values."""
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce",
    )


def display_table(table: pd.DataFrame, decimals: int = 2) -> pd.DataFrame:
    """Return a display copy with floating-point columns rounded for thesis-style reporting."""
    table = table.copy()
    float_cols = table.select_dtypes(include=["float", "float64", "float32"]).columns
    table[float_cols] = table[float_cols].round(decimals)
    return table


step_records = []


def record_step(step: str, before_rows: int | None, after_rows: int, note: str = "") -> None:
    removed = None if before_rows is None else before_rows - after_rows
    removed_percentage = None if before_rows in (None, 0) else round(removed / before_rows * 100, 2)
    retained_percentage = None if before_rows in (None, 0) else round(after_rows / before_rows * 100, 2)
    step_records.append(
        {
            "step": step,
            "rows_before": before_rows,
            "rows_after": after_rows,
            "rows_removed": removed,
            "removed_percentage": removed_percentage,
            "retained_percentage": retained_percentage,
            "note": note,
        }
    )

## 5. Load and normalize the raw data

This section loads the raw Kaggle/Crunchbase CSV file and standardizes column names for consistent processing.

In [52]:
df_raw = pd.read_csv(RAW_CSV_PATH, encoding="latin1")
df = normalize_column_names(df_raw)

print("Raw shape:", df_raw.shape)
print("Shape after column-name normalization:", df.shape)

required_columns = set(
    ["status", "founded_year", "country_code", "market", "funding_rounds"]
    + DATE_COLUMNS
    + FUNDING_AMOUNT_COLUMNS
)

missing_required_columns = sorted([col for col in required_columns if col not in df.columns])
if missing_required_columns:
    raise KeyError(f"Missing expected raw columns: {missing_required_columns}")

Raw shape: (54294, 39)
Shape after column-name normalization: (54294, 39)


## 6. Keep resolved outcomes and create the binary target

This section keeps only acquired and closed startups and creates the binary target variable used in the thesis.

In [53]:
before = len(df)

df = df[df["status"].isin(VALID_STATUSES)].copy()
df[TARGET_COLUMN] = df["status"].map(TARGET_MAP).astype(int)

record_step("Keep only acquired/closed observations and create binary target", before, len(df))

status_distribution = (
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)
status_distribution["percentage"] = (status_distribution["count"] / len(df) * 100).round(2)

target_distribution = (
    df[TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)
target_distribution["percentage"] = (target_distribution["count"] / len(df) * 100).round(2)

display(display_table(status_distribution))
display(display_table(target_distribution))

,status,count,percentage
0,acquired,3692,58.65
1,closed,2603,41.35


,target,count,percentage
0,1,3692,58.65
1,0,2603,41.35


## 7. Convert date and numeric variables

This section converts date fields to datetime format and converts funding-related variables to numeric values.

In [54]:
for col in DATE_COLUMNS:
    df[col] = pd.to_datetime(df[col], errors="coerce")

existing_funding_cols = [col for col in FUNDING_AMOUNT_COLUMNS if col in df.columns]

for col in existing_funding_cols:
    df[col] = clean_money_like_series(df[col])

# In this dataset, blank funding amount cells are treated as no recorded amount.
df[existing_funding_cols] = df[existing_funding_cols].fillna(0)

df["funding_rounds"] = pd.to_numeric(df["funding_rounds"], errors="coerce")
df["founded_year"] = pd.to_numeric(df["founded_year"], errors="coerce")

conversion_check = df[
    ["founded_year", "funding_rounds", "funding_total_usd"] + DATE_COLUMNS
].dtypes.rename("dtype").reset_index().rename(columns={"index": "column"})

display(display_table(conversion_check))

,column,dtype
0,founded_year,float64
1,funding_rounds,float64
2,funding_total_usd,float64
3,founded_at,datetime64[us]
4,first_funding_at,datetime64[us]
5,last_funding_at,datetime64[us]


## 8. Remove chronological inconsistencies

This section removes observations where the recorded founding date occurs after the first funding date.

In [55]:
before = len(df)

invalid_founded_after_first_funding = (
    df["founded_at"].notna()
    & df["first_funding_at"].notna()
    & (df["founded_at"] > df["first_funding_at"])
)

n_invalid_founded_after_first = int(invalid_founded_after_first_funding.sum())

df = df.loc[~invalid_founded_after_first_funding].copy()

record_step(
    "Remove observations where founded_at is later than first_funding_at",
    before,
    len(df),
    note=f"Removed {n_invalid_founded_after_first} observations.",
)

before = len(df)

invalid_first_after_last_funding = (
    df["first_funding_at"].notna()
    & df["last_funding_at"].notna()
    & (df["first_funding_at"] > df["last_funding_at"])
)

n_invalid_first_after_last = int(invalid_first_after_last_funding.sum())

df = df.loc[~invalid_first_after_last_funding].copy()

record_step(
    "Remove observations where first_funding_at is later than last_funding_at",
    before,
    len(df),
    note=f"Removed {n_invalid_first_after_last} observations.",
)

print("Rows where founded_at > first_funding_at:", n_invalid_founded_after_first)
print("Rows where first_funding_at > last_funding_at:", n_invalid_first_after_last)
print("Shape after chronological cleaning:", df.shape)

Rows where founded_at > first_funding_at: 447
Rows where first_funding_at > last_funding_at: 0
Shape after chronological cleaning: (5848, 40)


## 9. Apply the first-funding-year cutoff

This section applies the 1990 cutoff to first funding year, as used in the final modeling dataset.

In [56]:
df["first_funding_year"] = df["first_funding_at"].dt.year

before = len(df)

missing_first_funding_year = int(df["first_funding_year"].isna().sum())
before_cutoff = int((df["first_funding_year"] < MIN_FIRST_FUNDING_YEAR).sum())

df = df.loc[df["first_funding_year"] >= MIN_FIRST_FUNDING_YEAR].copy()

record_step(
    f"Apply first_funding_year >= {MIN_FIRST_FUNDING_YEAR}",
    before,
    len(df),
    note=(
        f"Rows with missing first_funding_year before cutoff: {missing_first_funding_year}; "
        f"rows below cutoff before filtering: {before_cutoff}."
    ),
)

print("Rows with missing first_funding_year before cutoff:", missing_first_funding_year)
print(f"Rows with first_funding_year < {MIN_FIRST_FUNDING_YEAR} before cutoff:", before_cutoff)
print("Shape after first-funding-year cutoff:", df.shape)

Rows with missing first_funding_year before cutoff: 0
Rows with first_funding_year < 1990 before cutoff: 3
Shape after first-funding-year cutoff: (5845, 41)


## 10. Handle missing values and apply the founded-year cutoff

This section imputes missing founding year from first funding year, fills selected categorical missing values, and applies the founded-year cutoff.

In [57]:
missing_before = (
    df[["founded_year", "country_code", "market"]]
    .isna()
    .sum()
    .rename("missing_before")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(display_table(missing_before))

# Missing founded_year is imputed from the first funding year.
df["founded_year"] = df["founded_year"].fillna(df["first_funding_at"].dt.year)

before = len(df)
below_founded_year_cutoff = int((df["founded_year"] < MIN_FOUNDING_YEAR).sum())

df = df.loc[df["founded_year"] >= MIN_FOUNDING_YEAR].copy()

record_step(
    f"Impute founded_year from first_funding_at and apply founded_year >= {MIN_FOUNDING_YEAR}",
    before,
    len(df),
    note=f"Rows below founded-year cutoff after imputation: {below_founded_year_cutoff}.",
)

# Missing categorical values are retained as an explicit category.
df["country_code"] = df["country_code"].fillna("other")
df["market"] = df["market"].fillna("other")

missing_after = (
    df[["founded_year", "country_code", "market"]]
    .isna()
    .sum()
    .rename("missing_after")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(display_table(missing_after))

print("Shape after missing-value handling and founded-year cutoff:", df.shape)

,column,missing_before
0,founded_year,1327
1,country_code,577
2,market,219


,column,missing_after
0,founded_year,0
1,country_code,0
2,market,0


Shape after missing-value handling and founded-year cutoff: (5704, 41)


## 11. Create engineered features

This section creates the funding-duration, funding-pace, single-round, funding-stage, and U.S. subgroup indicators.

In [58]:
df["funding_duration_days"] = (df["last_funding_at"] - df["first_funding_at"]).dt.days

df["avg_days_between_rounds"] = np.where(
    df["funding_rounds"] > 1,
    df["funding_duration_days"] / (df["funding_rounds"] - 1),
    0,
)

df["is_single_round"] = (df["funding_rounds"] == 1).astype(int)

for round_letter in ["a", "b", "c", "d", "e"]:
    amount_col = f"round_{round_letter}"
    indicator_col = f"has_round_{round_letter}"
    df[indicator_col] = (df[amount_col] > 0).astype(int)

df["is_usa"] = (df["country_code"] == "USA").astype(int)

engineered_features = [
    "funding_duration_days", "avg_days_between_rounds", "is_single_round",
    "has_round_a", "has_round_b", "has_round_c", "has_round_d", "has_round_e",
    "is_usa",
]

engineered_summary = df[engineered_features].describe().T.round(2)
display(display_table(engineered_summary))

,count,mean,std,min,25%,50%,75%,max
funding_duration_days,5704.00,366.33,635.02,0.00,0.00,0.00,559.00,5773.00
avg_days_between_rounds,5704.00,209.79,363.10,0.00,0.00,0.00,357.62,4700.00
is_single_round,5704.00,0.59,0.49,0.00,0.00,1.00,1.00,1.00
has_round_a,5704.00,0.27,0.44,0.00,0.00,0.00,1.00,1.00
has_round_b,5704.00,0.20,0.40,0.00,0.00,0.00,0.00,1.00
has_round_c,5704.00,0.11,0.32,0.00,0.00,0.00,0.00,1.00
has_round_d,5704.00,0.05,0.22,0.00,0.00,0.00,0.00,1.00
has_round_e,5704.00,0.02,0.14,0.00,0.00,0.00,0.00,1.00
is_usa,5704.00,0.68,0.47,0.00,0.00,1.00,1.00,1.00


## 12. Inspect sparse funding variables after preprocessing

This section calculates zero shares for funding amount variables after preprocessing to document sparsity-based feature refinement.

In [59]:
zero_share = (
    (df[FUNDING_AMOUNT_COLUMNS] == 0)
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
    .rename("zero_share_percent")
    .reset_index()
    .rename(columns={"index": "variable"})
)

display(display_table(zero_share))

,variable,zero_share_percent
0,round_h,100.00
1,post_ipo_debt,99.96
2,secondary_market,99.96
3,product_crowdfunding,99.95
4,equity_crowdfunding,99.93
5,round_g,99.93
6,post_ipo_equity,99.88
7,round_f,99.47
8,convertible_note,99.44
9,grant,99.02


## 13. Apply log1p to retained funding-scale variables

This section applies log1p transformation to the retained funding-scale variables used for modeling.

In [60]:
negative_counts = (df[LOG1P_COLUMNS] < 0).sum()

if (negative_counts > 0).any():
    raise ValueError(
        "Some retained funding amount columns contain negative values. "
        "Check the data before applying log1p."
    )

for col in LOG1P_COLUMNS:
    df[col] = np.log1p(df[col])

log_summary = df[LOG1P_COLUMNS].describe().T.round(2)
display(display_table(log_summary))

,count,mean,std,min,25%,50%,75%,max
funding_total_usd,5704.00,12.98,5.50,0.00,12.43,14.91,16.38,22.46
seed,5704.00,2.60,5.12,0.00,0.00,0.00,0.00,16.92
venture,5704.00,9.72,7.75,0.00,0.00,14.51,16.26,20.47
angel,5704.00,0.87,3.28,0.00,0.00,0.00,0.00,17.22
debt_financing,5704.00,1.31,4.19,0.00,0.00,0.00,0.00,20.91
private_equity,5704.00,0.34,2.40,0.00,0.00,0.00,0.00,20.46
undisclosed,5704.00,0.16,1.55,0.00,0.00,0.00,0.00,18.64


## 14. Build the final unsplit modeling dataset

This section selects the final predictors, target variable, subgroup variable, and temporal split variable.

In [61]:
final_columns = full_features + [TARGET_COLUMN, SUBGROUP_COLUMN, TEMPORAL_SPLIT_COLUMN]

missing_final_columns = [col for col in final_columns if col not in df.columns]
if missing_final_columns:
    raise KeyError(f"Missing expected final columns: {missing_final_columns}")

df_preprocessed = df[final_columns].copy()

binary_columns = [
    TARGET_COLUMN, SUBGROUP_COLUMN,
    "is_single_round", "has_round_a", "has_round_b", "has_round_c", "has_round_d", "has_round_e",
]

for col in binary_columns:
    df_preprocessed[col] = df_preprocessed[col].astype(int)

final_missing_counts = (
    df_preprocessed
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

target_distribution_final = (
    df_preprocessed[TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("target")
    .reset_index(name="count")
)
target_distribution_final["percentage"] = (target_distribution_final["count"] / len(df_preprocessed) * 100).round(2)

print("Final preprocessed unsplit dataset shape:", df_preprocessed.shape)
print("Expected thesis shape: 5,704 observations and 22 columns")
print("Number of predictors:", len(full_features))

display(display_table(final_missing_counts))
display(display_table(target_distribution_final))
display(display_table(df_preprocessed.head()))

Final preprocessed unsplit dataset shape: (5704, 22)
Expected thesis shape: 5,704 observations and 22 columns
Number of predictors: 19


,column,missing_count
0,funding_total_usd,0
1,seed,0
2,is_usa,0
3,target,0
4,market,0
5,country_code,0
6,has_round_e,0
7,has_round_d,0
8,has_round_c,0
9,has_round_b,0


,target,count,percentage
0,1,3387,59.38
1,0,2317,40.62


,funding_total_usd,seed,venture,angel,debt_financing,private_equity,undisclosed,founded_year,funding_rounds,funding_duration_days,avg_days_between_rounds,is_single_round,has_round_a,has_round_b,has_round_c,has_round_d,has_round_e,country_code,market,target,is_usa,last_funding_at
0,14.38,14.38,0.00,0.00,0.00,0.00,0.00,2012.00,1.00,0,0.00,1,0,0,0,0,0,USA,News,1,1,2012-06-30
6,15.41,0.00,0.00,0.00,0.00,0.00,15.41,2007.00,1.00,0,0.00,1,0,0,0,0,0,ARG,Advertising,0,0,2007-01-16
18,13.12,13.12,0.00,0.00,0.00,0.00,0.00,2009.00,1.00,0,0.00,1,0,0,0,0,0,other,Marketplaces,1,0,2009-05-15
27,14.04,13.53,13.12,0.00,0.00,0.00,0.00,2011.00,2.00,28,28.00,0,0,0,0,0,0,USA,Analytics,1,1,2011-11-30
30,10.82,10.82,0.00,0.00,0.00,0.00,0.00,2009.00,1.00,0,0.00,1,0,0,0,0,0,USA,Curated Web,0,1,2009-04-01


## 15. Save outputs

This section saves the unsplit modeling dataset and preprocessing summary tables for later notebooks.

In [62]:
step_summary = display_table(pd.DataFrame(step_records))

# The saved modeling dataset keeps the full numeric precision needed for later modeling.
df_preprocessed.to_csv(PREPROCESSED_DATA_PATH, index=False)

# Save summary tables generated above.
step_summary.to_csv(PREPROCESSING_STEPS_PATH, index=False)
zero_share.to_csv(ZERO_SHARE_PATH, index=False)
feature_block_table.to_csv(FEATURE_BLOCKS_PATH, index=False)